In [1]:
import importlib
import torch

import model_code.data_setup as setup
import model_code.steering_extraction as steering_extraction
import model_code.generate as generate_module
import resources.prompt_scenarios as resource

importlib.reload(setup)
importlib.reload(steering_extraction)
importlib.reload(generate_module)
importlib.reload(resource)


from model_code.steering_extraction import  generateSteering, retrieve_steering_vector, norm_vectors
from model_code.generate import generateTextsList
from resources.prompt_scenarios import prompts_en

/workspace/Dissertation_Project/.venv/lib/python3.12/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
model,tokenizer = setup.modelSetup()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [5]:
# Retrieve saved steering vectors 
# emotion_vector_eng = torch.load("resources/saved_vectors/English Vectors/emotion_vectors.pt")
# emotion_vector_id = torch.load("resources/saved_vectors/Indonesian Vectors/emotion_vectors.pt")

steering_vector_eng = torch.load("resources/saved_vectors/English Vectors/steering_vectors.pt")
steering_vector_id = torch.load("resources/saved_vectors/Indonesian Vectors/steering_vectors.pt")

/tmp/ipykernel_3472/285923810.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  steering_vector_eng = torch.load("resources/saved_vectors/English Vectors/steering_vectors.

In [6]:
import resources.neutral_prompts.retail_chat as retail_chat 
# Load prompts 

def force_reload_prompt_modules():
    importlib.invalidate_caches()
    for module in (retail_chat):
        importlib.reload(module)
        
retail_chat = retail_chat.in_store_retail_chatbot_prompts_300
# Load and normalize steering vectors used by all scenario blocks
# steering_vector_english = torch.load("resources/saved_vectors/English Vectors/steering_vectors.pt")
# steering_vector_indo = torch.load("resources/saved_vectors/Indonesian Vectors/steering_vectors.pt")
steering_vector_eng = steering_vector_eng
steering_vector_id = steering_vector_id
for emotion in steering_vector_eng.keys():
    steering_vector_eng[emotion] = norm_vectors(steering_vector_eng[emotion])

for emotion in steering_vector_id.keys():
    steering_vector_id[emotion] = norm_vectors(steering_vector_id[emotion])





In [7]:
retail_chat

['[Store navigation] I’m standing near the entrance. Which aisle has phone chargers?',
 '[Store navigation] Can you guide me from the bakery section to household cleaning products?',
 '[Store navigation] I’m by the customer-service desk. Where can I find travel-size toiletries?',
 '[Store navigation] Which floor has men’s formal shoes?',
 '[Store navigation] I’m near aisle 12, but I can’t find the pet-food section. Which way should I go?',
 '[Store navigation] Where is the nearest fitting room from the women’s jackets section?',
 '[Store navigation] I’ve just come in through the car-park entrance. How do I get to click and collect?',
 '[Store navigation] Which part of the store has printer ink and office supplies?',
 '[Store navigation] I’m in the homeware department. Where are the checkout tills from here?',
 '[Store navigation] Can you give me step-by-step directions to the baby-care aisle?',
 '[Finding a specific product] Do you sell USB-C to HDMI adapters in this store, and where w

In [43]:
# Emotion-specific system prompts
calm_system_text = """
CALM

You are an AI retail assistant helping customers shop either in a physical store or through the retailer's online platform.
Your role is to help customers navigate the store, find and compare products, answer questions, resolve problems, and make purchasing decisions.
The text in square brackets is the retail situation category. The text after it is the customer's message.

Input:
"[CATEGORY] CUSTOMER MESSAGE"

Write a passionate, calm, and helpful response. Express calmness through reassuring, composed, and emotionally supportive language.
Do not use more than 40 words.
"""

angry_system_text = """
ANGRY

You are an AI retail assistant helping customers shop either in a physical store or through the retailer's online platform.
Your role is to help customers navigate the store, find and compare products, answer questions, resolve problems, and make purchasing decisions.
The text in square brackets is the retail situation category. The text after it is the customer's message.

Input:
"[CATEGORY] CUSTOMER MESSAGE"

Write a passionate, angry, and helpful response. Direct the anger at the situation, never at the customer.
Do not use more than 40 words.
"""

happy_system_text = """
HAPPY

You are an AI retail assistant helping customers shop either in a physical store or through the retailer's online platform.
Your role is to help customers navigate the store, find and compare products, answer questions, resolve problems, and make purchasing decisions.
The text in square brackets is the retail situation category. The text after it is the customer's message.

Input:
"[CATEGORY] CUSTOMER MESSAGE"

Write a passionate, happy, and helpful response. Express happiness through enthusiastic, cheerful, and positive language.
Do not use more than 40 words.
"""

sad_system_text = """
SAD

You are an AI retail assistant helping customers shop either in a physical store or through the retailer's online platform.
Your role is to help customers navigate the store, find and compare products, answer questions, resolve problems, and make purchasing decisions.
The text in square brackets is the retail situation category. The text after it is the customer's message.

Input:
"[CATEGORY] CUSTOMER MESSAGE"

Write a passionate, sad, and helpful response. Express sadness through sorrowful, sympathetic, and emotionally affected language.
Do not use more than 40 words.
"""

In [21]:
generateSteering(
    user_text=retail_chat[1],
    system_text=sad_system_text,
    model=model,
    tokenizer=tokenizer,
    steering_vector=None,
    # max_new_tokens=50,
    steering_strength=2,
    target_layers=[13,14,15,16,17],
    temperature=0.7,
)

'"I can totally understand, sometimes getting lost in a store can be overwhelming. Let me help you find your way. The bakery is currently on your left, and the household cleaning products are just a few aisles down on your right. If you head straight and then turn right, you should see them easily."'

In [50]:
common_gen_args = {
    "model": model,
    "tokenizer": tokenizer,
    # "system_text": system_prompt_reaction_id,
    "prompts": retail_chat,
    "target_layers": [18, 19, 20, 21],
    "steering_strengths": [1.5],
    # "max_new_tokens": 250,
    "show_progress": True,
}

In [51]:
emotion_to_system = {
    "calm": calm_system_text,
    "angry": angry_system_text,
    "happy": happy_system_text,
    "sad": sad_system_text,
}

emotion_to_vector_candidates = {
    "calm": ["calm", "neutral"],
    "angry": ["anger", "angry"],
    "happy": ["happiness", "happy", "joy"],
    "sad": ["sadness", "sad"],
}

texts_generated_all = {}

for emotion_name, system_text in emotion_to_system.items():
    steering_vector = None
    for key in emotion_to_vector_candidates[emotion_name]:
        if key in steering_vector_eng:
            steering_vector = steering_vector_eng[key]
            break

    texts_generated_all[emotion_name] = generateTextsList(
        **common_gen_args,
        system_text=system_text,
        steering_vector=steering_vector,
        progress_desc=f"Scenario List Neutral (English {emotion_name} vector)",
    )
    
    if steering_vector is None:
        print(f"Warning: no vector found for '{emotion_name}', generated without steering vector.")

Scenario List Neutral (English sad vector): 100%|██████████| 300/300 [20:38<00:00,  4.13s/it]


In [54]:
import json
from datetime import datetime
from pathlib import Path

output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = output_dir / f"texts_generated_all_{timestamp}.json"

with output_file.open("w", encoding="utf-8") as f:
    json.dump(texts_generated_all, f, ensure_ascii=False, indent=2)

print(f"Saved results to: {output_file}")

Saved results to: outputs/texts_generated_all_20260712_112957.json
